# Named Entity Recognition (NER) Lab #14

## Pre-Lab Task

### 1) Define Named Entity Recognition?

**Answer:**  
Named Entity Recognition (NER) is a subtask of Information Extraction that seeks to locate and classify named entities mentioned in unstructured text into pre-defined categories such as person names, organizations, locations, medical codes, time expressions, quantities, monetary values, percentages, etc.

### 2) What are the use cases of Named Entity Recognition?

**Answer:**  
*   **Content Classification:** Automatically tagging news articles or blog posts with entities to improve search and recommendations.
*   **Customer Support:** Extracting product names, order IDs, or locations from customer queries to route them efficiently.
*   **Resume Screening:** Extracting skills, universities, and previous employers from resumes.
*   **Search Engines:** Enhancing search results by understanding the entities within a query (e.g., recognizing "Apple" as a company vs. a fruit).

### 3) What are the applications of Named Entity Recognition?

**Answer:**  
*   **Question Answering Systems:** Identifying the "who," "where," and "when" in a question to provide precise answers.
*   **Information Extraction for Finance:** Extracting company names and monetary values from financial reports.
*   **Healthcare:** Extracting drug names, diseases, and patient identifiers from clinical notes.
*   **Semantic Search:** Building knowledge graphs by linking extracted entities.

## In Lab Task

### 1) Srinivas Scenario: Organization-Location Connectivity

Srinivas may be keen on the connection among organizations and areas. Given an organization, we might want to have the option to recognize the areas where it works together; on the other hand, given an area, we might want to find which organizations work together in that area. Our data is in tabular form, then answering these queries is straightforward.

In [9]:
import spacy
from spacy.pipeline import EntityRuler

# Load the English NER model
nlp = spacy.load("en_core_web_sm")

# Add custom entity patterns for the specific companies and Indian cities
ruler = nlp.add_pipe("entity_ruler", before="ner")
patterns = [
    {"label": "ORG", "pattern": "TCS"},
    {"label": "ORG", "pattern": "INFOCEPT"},
    {"label": "ORG", "pattern": "WIPRO"},
    {"label": "ORG", "pattern": "AMAZON"},
    {"label": "ORG", "pattern": "INTEL"},
    {"label": "GPE", "pattern": "PUNE"},
    {"label": "GPE", "pattern": "HYDERABAD"},
]
ruler.add_patterns(patterns)

# Representing the organization-location-year data as natural language text
texts = [
    "TCS is headquartered in PUNE and was founded in 1968.",
    "INFOCEPT is headquartered in PUNE and was founded in 1972.",
    "WIPRO is headquartered in PUNE and was founded in 1945.",
    "AMAZON is headquartered in HYDERABAD and was founded in 1994.",
    "INTEL is headquartered in HYDERABAD and was founded in 1968.",
]

# Extract entities using NER and build lookup dictionaries
location_org_map = {}  # location -> list of orgs
org_year_map = {}      # org -> founding year

print("NER Entity Extraction:\n")
for text in texts:
    doc = nlp(text)
    entities = [(ent.text, ent.label_) for ent in doc.ents]
    print(f"Text: {text}")
    print(f"Entities: {entities}\n")

    org = location = year = None
    for ent in doc.ents:
        if ent.label_ == "ORG":
            org = ent.text
        elif ent.label_ == "GPE":
            location = ent.text
        elif ent.label_ == "DATE":
            year = ent.text

    if org and location:
        location_org_map.setdefault(location, []).append(org)
    if org and year:
        org_year_map[org] = year

NER Entity Extraction:

Text: TCS is headquartered in PUNE and was founded in 1968.
Entities: [('TCS', 'ORG'), ('PUNE', 'GPE'), ('1968', 'DATE')]

Text: INFOCEPT is headquartered in PUNE and was founded in 1972.
Entities: [('INFOCEPT', 'ORG'), ('PUNE', 'GPE'), ('1972', 'DATE')]

Text: WIPRO is headquartered in PUNE and was founded in 1945.
Entities: [('WIPRO', 'ORG'), ('PUNE', 'GPE'), ('1945', 'DATE')]

Text: AMAZON is headquartered in HYDERABAD and was founded in 1994.
Entities: [('AMAZON', 'ORG'), ('HYDERABAD', 'GPE'), ('1994', 'DATE')]

Text: INTEL is headquartered in HYDERABAD and was founded in 1968.
Entities: [('INTEL', 'ORG'), ('HYDERABAD', 'GPE'), ('1968', 'DATE')]



#### Queries:
**1) Which associations work in HYDERABAD?**

In [2]:
# Query 1: Which organizations work in HYDERABAD?
# Use the NER-extracted location_org_map to answer
hyderabad_orgs = location_org_map.get("HYDERABAD", [])
print(f"Organizations in HYDERABAD: {', '.join(hyderabad_orgs)}")

Organizations in HYDERABAD: AMAZON, INTEL


**2) Which organizations operate in PUNE?**

In [3]:
# Query 2: Which organizations operate in PUNE?
# Use the NER-extracted location_org_map to answer
pune_orgs = location_org_map.get("PUNE", [])
print(f"Organizations in PUNE: {', '.join(pune_orgs)}")

Organizations in PUNE: TCS, INFOCEPT, WIPRO


**3) In which year AMAZON and TCS is formed?**

In [4]:
# Query 3: In which year were AMAZON and TCS formed?
# Use the NER-extracted org_year_map to answer
amazon_year = org_year_map.get("AMAZON", "Not found")
tcs_year = org_year_map.get("TCS", "Not found")
print(f"AMAZON was formed in: {amazon_year}")
print(f"TCS was formed in: {tcs_year}")

AMAZON was formed in: 1994
TCS was formed in: 1968


## Task 2: NLP Sentence Analysis

Sentence: *"European authorities fined Google a record $5.1 billion on Wednesday for abusing its power in the mobile phone market and ordered the company to alter its practices."*

### a) Import the libraries

In [5]:
import nltk
from nltk import word_tokenize, pos_tag, ne_chunk, RegexpParser

# Download necessary NLTK datasets
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('maxent_ne_chunker')
nltk.download('words')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/rajaramkankipati/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/rajaramkankipati/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     /Users/rajaramkankipati/nltk_data...
[nltk_data]   Package maxent_ne_chunker is already up-to-date!
[nltk_data] Downloading package words to
[nltk_data]     /Users/rajaramkankipati/nltk_data...
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/rajaramkankipati/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/rajaramkankipati/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already

True

### b) Apply word tokenization and part-of-speech tagging to the sentence

In [6]:
sentence = "European authorities fined Google a record $5.1 billion on Wednesday for abusing its power in the mobile phone market and ordered the company to alter its practices."

tokens = word_tokenize(sentence)
pos_tags = pos_tag(tokens)

print("Tokens:", tokens)
print("\nPOS Tags:", pos_tags)

Tokens: ['European', 'authorities', 'fined', 'Google', 'a', 'record', '$', '5.1', 'billion', 'on', 'Wednesday', 'for', 'abusing', 'its', 'power', 'in', 'the', 'mobile', 'phone', 'market', 'and', 'ordered', 'the', 'company', 'to', 'alter', 'its', 'practices', '.']

POS Tags: [('European', 'JJ'), ('authorities', 'NNS'), ('fined', 'VBD'), ('Google', 'NNP'), ('a', 'DT'), ('record', 'NN'), ('$', '$'), ('5.1', 'CD'), ('billion', 'CD'), ('on', 'IN'), ('Wednesday', 'NNP'), ('for', 'IN'), ('abusing', 'VBG'), ('its', 'PRP$'), ('power', 'NN'), ('in', 'IN'), ('the', 'DT'), ('mobile', 'JJ'), ('phone', 'NN'), ('market', 'NN'), ('and', 'CC'), ('ordered', 'VBD'), ('the', 'DT'), ('company', 'NN'), ('to', 'TO'), ('alter', 'VB'), ('its', 'PRP$'), ('practices', 'NNS'), ('.', '.')]


### c) Create a chunk parser and test it on our sentence

In [7]:
# Defining a simple grammar for NP (Noun Phrase) chunking
grammar = "NP: {<DT>?<JJ>*<NN>+}"
cp = RegexpParser(grammar)
result = cp.parse(pos_tags)

print(result)
# result.draw() # To visualize (requires Ghostscript)

(S
  European/JJ
  authorities/NNS
  fined/VBD
  Google/NNP
  (NP a/DT record/NN)
  $/$
  5.1/CD
  billion/CD
  on/IN
  Wednesday/NNP
  for/IN
  abusing/VBG
  its/PRP$
  (NP power/NN)
  in/IN
  (NP the/DT mobile/JJ phone/NN market/NN)
  and/CC
  ordered/VBD
  (NP the/DT company/NN)
  to/TO
  alter/VB
  its/PRP$
  practices/NNS
  ./.)


### d) Identify nationalities or religious or political groups, organization, date and money in the given sentence

In [8]:
# Using NLTK's ne_chunk to identify named entities
ner_results = ne_chunk(pos_tags)

print("Named Entities Identified:")
for chunk in ner_results:
    if hasattr(chunk, 'label'):
        print(f"{chunk.label()}: {' '.join(c[0] for c in chunk)}")

print("\nManual Mapping based on constraints:")
# Note: 'European' often gets tagged as GPE or JJ. 
# $5.1 billion and Wednesday are usually handled by more specific parsers.
print("Nationalities/Groups: European")
print("Organization: Google")
print("Date: Wednesday")
print("Money: $5.1 billion")

Named Entities Identified:
GPE: European
PERSON: Google

Manual Mapping based on constraints:
Nationalities/Groups: European
Organization: Google
Date: Wednesday
Money: $5.1 billion
